In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import importlib

import utils

results_dir = Path() / "results" / "sphere_tracing_ray_split_thresholds"
figures_dir = Path() / "figures"

In [ ]:
cache = True # Set to True to skip already benchmarked configurations
methods = ["iarap"]
num_rays = [100, 1000, 10000, 100000]
ray_split_thresholds = [0, 0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5] # Length thresholds for ray splitting

results_dir.mkdir(parents=True, exist_ok=True)

for method in methods:
    for num_ray in num_rays:
        for ray_split_threshold in ray_split_thresholds:
            output_path = results_dir / f"{method}_{num_ray}_{ray_split_threshold}.csv"
            if output_path.exists() and cache:
                print(f"Skipping {method} with {num_ray} rays (cached).")
                continue
            
            utils.call_script(Path().absolute() / "benchmark_sphere_tracing.py", method=method, num_rays=num_ray, dataset="neural_iarap", num_runs=5, 
                              ray_split_threshold=ray_split_threshold,
                              tag=f"{ray_split_threshold}", output=output_path)

In [ ]:
importlib.reload(utils)

# Load and merge all dataframes
dataframes = [pd.read_csv(f) for f in results_dir.glob("*.csv")]
df_raw = pd.concat(dataframes, ignore_index=True)

# Compute averages and standard deviations over runs, and merge them into a single dataframe
df_avg_by_sdf = df_raw.groupby(["method", "num_rays", "dataset", "sdf_name", "tag"]).agg({"elapsed_time": "mean", "peak_memory": "mean"}).reset_index()
df_std_by_sdf = df_raw.groupby(["method", "num_rays", "dataset", "sdf_name", "tag"]).agg({"elapsed_time": "std", "peak_memory": "std"}).reset_index()
df_by_sdf = pd.merge(df_avg_by_sdf, df_std_by_sdf, on=["method", "num_rays", "dataset", "sdf_name", "tag"], suffixes=("_avg", "_std"))

# Compute averages over SDFs, and merge them into a single dataframe
df = df_by_sdf.groupby(["method", "num_rays", "tag"]).agg({"elapsed_time_avg": "mean", "peak_memory_avg": "mean"}).reset_index()

# Only use seaborn colors for the plots
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.style.library["seaborn-v0_8"]['axes.prop_cycle'])

fig = plt.figure(figsize=(6, 5))

utils.set_figure_title_with_specs(fig, benchmark_name="Sphere Tracing (Ray Split Thresholds)")

gs = fig.add_gridspec(2, 1, hspace=0.08)

ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor('#EAEAF2')
ax1.set_ylabel("Runtime (ms)", fontsize=12)
ax1.tick_params(labelbottom=False)

ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
ax2.set_facecolor('#EAEAF2')
ax2.set_xlabel("Ray split threshold", fontsize=12)
ax2.set_ylabel("Peak memory (MiB)", fontsize=12)

# Each num_rays gets its own line in the plot
for num_rays in df["num_rays"].unique():
    df_subset = df[df["num_rays"] == num_rays]
    ax1.plot(df_subset["tag"], df_subset["elapsed_time_avg"], marker="o", linestyle="-", label=f"{num_rays} Rays")
    ax2.plot(df_subset["tag"], df_subset["peak_memory_avg"], marker="o", linestyle="-", label=f"{num_rays} Rays")

ax2.legend()

# Mark the default value with a vertical line
import inspect
from fast_implicit_uniform_sampling import sphere_trace_all_intersections
default_threshold = inspect.signature(sphere_trace_all_intersections).parameters["ray_split_threshold"].default
for i, ax in enumerate([ax1, ax2]):
    ax.axvline(x=default_threshold, color="black", alpha=0.2, linestyle="--", label=f"Default Threshold ({default_threshold})", zorder=0)
    if i == 1:
        ax.annotate(f"Default ({default_threshold})", xy=(default_threshold, ax.get_ylim()[1] // 2), xytext=(5, -5), textcoords='offset points', color="black", alpha=0.5, fontsize=10, rotation=90, va='center')

figures_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figures_dir / "benchmark_sphere_tracing_ray_split_thresholds.png", dpi=150, bbox_inches="tight")